# Lecture 2: SQL JOINs & Pandas Data Manipulation

In this lecture, we'll explore how to combine data from multiple tables using **SQL JOINs** and how to filter and manipulate DataFrames using **Pandas**. We'll learn the differences between `.loc` and `.iloc`, understand how to apply boolean masks for filtering, and discover important indexing patterns to avoid common pitfalls when working with filtered data.

### Install dependencies

In [ ]:
%pip install pandas

We define a helper function `run_query()` to easily run SQL commands and return the results as a Pandas DataFrame.

### Import libraries

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
db_filename = 'chinook.db'

def run_query(q):
    # This creates a connection, runs the query, and closes the connection automatically
    with sqlite3.connect(db_filename) as conn:
        # Check if this is a SELECT query or a write operation
        if q.strip().upper().startswith('SELECT'):
            return pd.read_sql(q, conn)
        else:
            # For INSERT, UPDATE, DELETE, etc.
            conn.execute(q)
            conn.commit()
            return None

# Test the connection
try:
    print("Connecting to database...")
    tables = run_query("SELECT name FROM sqlite_master WHERE type='table';")
    print("Success! Tables found:")
    print(tables)
except Exception as e:
    print(f"Error: {e}")
    print("Double-check that 'chinook.db' is in the same directory as this script. ")

### Database Tables
A database most often contains one or more tables. Each table is identified by a name (e.g., "album" or "artist") and contains records (rows) with data.

We can see the list of tables in our database below:

In [ ]:
run_query("SELECT name FROM sqlite_master WHERE type='table';")

### 1. JOIN
We can assign to each row an id and connect entities through their ids.

In [ ]:
run_query(
"""SELECT
    artist.name AS Artist,
    album.title AS Album
FROM
    album
INNER JOIN
    artist ON album.artist_id = artist.artist_id;""").head()

### 2. Different Types of SQL JOINs
Here are the different types of the JOINs in SQL:
* (INNER) JOIN: Returns records that have matching values in both tables
* LEFT (OUTER) JOIN: Returns all records from the left table, and the matched records from the right table
* RIGHT (OUTER) JOIN: Returns all records from the right table, and the matched records from the left table
* FULL (OUTER) JOIN: Returns all records when there is a match in either left or right table

#### LEFT JOIN
The LEFT JOIN keyword returns all records from the left table (table1), and the matching records from the right table (table2). If there is no match on table2 then it will present NULL values for table2 columns.


In [ ]:
run_query(
"""SELECT
    artist.name as Artist,
    album.title as Album
FROM
    artist
LEFT JOIN
    album 
ON album.artist_id = artist.artist_id;""").head()

In [ ]:
run_query(
"""SELECT
    artist.name as Artist,
    album.title as Album
FROM
    artist
LEFT JOIN
    album 
ON album.artist_id = artist.artist_id
WHERE 
Album is NULL;""").head()

### 3. Multiple JOINS

We can join tables multiple times. This is useful when you need to combine data from 3 or more tables.

In this example, we join the **artist**, **album**, and **track** tables together:
1. First, we join `artist` with `album` using `artist_id`
2. Then, we join the result with `track` using `album_id`

This gives us artist names, album titles, and the tracks within each album.


In [ ]:
run_query("""
SELECT
    artist.name AS Artist,
    album.title AS Album,
    track.name AS Track
FROM
    artist
JOIN
    album ON artist.artist_id = album.artist_id
JOIN
    track ON album.album_id = track.album_id
LIMIT 10""")


You can also use multiple LEFT JOINs to include artists, albums, and genres even if some data is missing:


In [ ]:
run_query("""
SELECT
    artist.name AS Artist,
    album.title AS Album,
    track.name AS Track,
    genre.name AS Genre
FROM
    artist
LEFT JOIN
    album ON artist.artist_id = album.artist_id
LEFT JOIN
    track ON album.album_id = track.album_id
LEFT JOIN
    genre ON track.genre_id = genre.genre_id
LIMIT 10""")


## Pandas: .loc vs. .iloc

When working with DataFrames in Pandas, there are two primary methods for selecting rows and columns: `.loc` and `.iloc`. It's crucial to understand the difference between them, as they work in fundamentally different ways.

### Key Differences:

| Feature | `.loc` | `.iloc` |
|---------|--------|--------|
| **Selection Method** | By **label** (index name) | By **integer position** |
| **Index Type** | Works with index labels (strings, dates, custom names) | Works with positions (0, 1, 2, ...) |
| **Includes End** | Inclusive (includes the end range) | Exclusive (excludes the end range) |
| **Syntax** | `.loc[row_label, column_label]` | `.iloc[row_position, column_position]` |

### .iloc: Integer Location
`.iloc` uses integer-based positioning, like Python lists. It's useful when you want to select rows or columns by their position.


In [ ]:
# First, let's load the Track data again for demonstration
df = pd.read_csv('data/Track.csv', header=None)
df.columns = ["TrackId","Name","AlbumId","MediaTypeId","GenreId","Composer","Milliseconds","Bytes","UnitPrice"]


### .loc: Label-Based Location
`.loc` uses the index labels to select data. By default, the index is 0, 1, 2, etc., but `.loc` treats these as **labels**, not positions. Importantly, `.loc` is **inclusive on both ends** when using slices.


In [ ]:
print("=== ILOC (exclusive on right end) ===")
print(df.iloc[1:4, [1, 5]])  # Positions 1 to 3 (4 is excluded)

print("\n=== LOC (inclusive on both ends) ===")
print(df.loc[1:4, ['Name', 'Composer']])  # Labels 1 to 4 (both included!)


### When to Use Each

**Use `.iloc` when:**
- You want to select by position (like in lists)
- You know the row/column number
- You prefer Python-style exclusive end slicing

**Use `.loc` when:**
- You want to select by index label or column name
- You need to filter based on conditions (boolean indexing)
- You prefer inclusive slicing for clarity
- Working with custom indices (dates, strings, etc.)

**Best Practice:** `.loc` is generally preferred in modern Pandas code for its flexibility and readability, especially when combining filtering conditions.


### Important: Index Issues After Masking

When you use `.loc` to filter data with a boolean mask, the resulting DataFrame **retains the original index labels**. This can cause unexpected behavior or indexing errors if you later try to access rows by position or reset index-dependent operations.

**The Problem:**
- When you filter a DataFrame, the indices don't get renumbered (0, 1, 2, ...)
- Instead, the original indices are preserved
- This can cause confusion and errors when you try to work with the filtered data

**The Solution:**
Use `.reset_index(drop=True)` to renumber the rows from 0 to n-1 and discard the old index.


In [ ]:
# Example: Notice the index preserves original row numbers after filtering
# First, find a composer that exists in the data
sample_composer = df['Composer'].dropna().iloc[0]
print(f"Filtering by composer: {sample_composer}")

filtered_tracks = df.loc[df['Composer'] == sample_composer, ['Name', 'Composer']]
print("\n--- Filtered data (WITH old indices) ---")
print(filtered_tracks.head())

print("\n--- After reset_index(drop=True) ---")
filtered_tracks_clean = filtered_tracks.reset_index(drop=True)
print(filtered_tracks_clean.head())

print("\n--- Now accessing by iloc works as expected ---")
print("First row:", filtered_tracks_clean.iloc[0]['Name'])
